# 📘 Notebook 17: Hash Tables

### Course: *From Zero to Data Structures & Algorithms in Python*
**Instructor:** Ioannis Tsioulis

*Module D: Data Structures · Notebook 3 of 4 in this module · (17 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain the idea behind a **hash table**: computing a position instead of searching for it
- Describe what a **hash function** does and what makes a good one
- Explain what a **collision** is and resolve it with **chaining**
- Implement a working hash table as a class, with `put`, `get` and `remove`
- Explain the role of the **load factor** and of **resizing**
- Relate all of this to Python's `dict` and `set`, and use them to turn O(n²) solutions into O(n)

> **Prerequisites:** Notebooks 05 (dictionaries), 07 (classes), 11 (Big O) and 13 (searching).
>
> 🔭 **A promise kept.** In Notebook 05 we said that a dictionary finds a value "almost instantly, however large it is", and that Module D would explain how. In Notebook 11 we measured a set answering several hundred times faster than a list. This is the notebook where the mystery is solved.

## 1. The problem: finding things fast

So far we have two ways to find a value.

| Method | Cost | Requirement |
|---|---|---|
| linear search | O(n) | none |
| binary search | O(log n) | the data must be kept sorted |

Can we do better than O(log n)? Astonishingly, yes: **O(1)**, a cost that does not grow with the amount of data at all.

### Intuition first
A cloakroom at a theatre. You hand over your coat and receive a ticket with a number. The attendant hangs the coat on the hook with that number. When you return, the attendant does not search through the coats. The **number on your ticket says exactly where to go**.

Finding the coat takes the same time whether the cloakroom holds 10 coats or 10,000.

A hash table does the same thing, with one twist: instead of being handed a ticket, it **computes the number from the key itself**.

### The plan
1. Keep an ordinary list of fixed size. Reading `slots[i]` is O(1), as we know.
2. Turn each key into an index of that list by means of a function.
3. Store the value at that index. To find it later, compute the same index again and look there.

No searching is involved at any point.

## 2. Hash functions

### Formal definition
A **hash function** takes a key and returns a whole number. To turn that number into a valid index for a list of `size` slots, we take the remainder of the division by `size`, with the `%` operator of Notebook 01.

A good hash function must be:

- **deterministic**: the same key always gives the same number;
- **fast** to compute;
- **well spread**: different keys should land on different indices as evenly as possible.

### A first, simple hash function
Every character has a numeric code, which `ord()` returns. Let us add up the codes of the characters of a string.

In [ ]:
print(ord("a"), ord("b"), ord("A"))

def simple_hash(key):
    total = 0
    for char in key:
        total += ord(char)
    return total

print(simple_hash("cat"))
print(simple_hash("dog"))

In [ ]:
size = 10          # our table has 10 slots, with indices 0 to 9

for word in ["cat", "dog", "bird", "fish", "horse"]:
    index = simple_hash(word) % size
    print(f"{word:<6} hash = {simple_hash(word):<4} -> slot {index}")

Each word has been turned into a slot number between 0 and 9, **without looking at any other word**. That is the whole trick.

### Storing and finding by computed index

In [ ]:
slots = [None] * size

# Store some ages
slots[simple_hash("cat") % size] = 4
slots[simple_hash("dog") % size] = 7
slots[simple_hash("horse") % size] = 12
print(slots)

# Find the age of the dog: compute the index again, and look there
print("dog ->", slots[simple_hash("dog") % size])

## 3. Collisions

### The problem
There are infinitely many possible keys and only a handful of slots. Sooner or later two different keys will be sent to the **same** slot. This is called a **collision**, and it cannot be avoided, only managed.

Our simple hash function has an obvious weakness: it ignores the **order** of the letters, so all anagrams collide.

In [ ]:
for word in ["listen", "silent", "enlist"]:
    print(f"{word:<7} hash = {simple_hash(word)} -> slot {simple_hash(word) % size}")

If we stored `"listen"` and then `"silent"` with the code of the previous section, the second would silently **overwrite** the first.

### The solution: chaining
Instead of holding a single value, each slot holds a small **list** of `(key, value)` pairs, called a **bucket**. Colliding keys simply share a bucket.

```
slot 0: []
slot 1: [("cat", 4)]
slot 2: [("dog", 7), ("god", 99)]        <- a collision, both kept
slot 3: []
```

To find a key we compute its slot and then search **only that bucket**. As long as the buckets stay short, this short search costs almost nothing.

## 4. Building a hash table

We now have every ingredient. The class below has the same three operations as a Python dictionary.

| Our method | Python dictionary equivalent |
|---|---|
| `table.put(key, value)` | `d[key] = value` |
| `table.get(key)` | `d.get(key)` |
| `table.remove(key)` | `del d[key]` |

Each method starts in the same way: compute the index of the bucket, then work inside that bucket only.

In [ ]:
class HashTable:
    def __init__(self, size=10):
        self.size = size
        self.buckets = [[] for i in range(size)]     # one empty list per slot
        self.count = 0

    def _index(self, key):
        return simple_hash(key) % self.size

    def put(self, key, value):
        bucket = self.buckets[self._index(key)]
        for i in range(len(bucket)):
            if bucket[i][0] == key:              # the key exists: update its value
                bucket[i] = (key, value)
                return
        bucket.append((key, value))              # a new key: add it to the bucket
        self.count += 1

    def get(self, key):
        bucket = self.buckets[self._index(key)]
        for stored_key, value in bucket:
            if stored_key == key:
                return value
        return None                              # the key is not in the table

    def remove(self, key):
        bucket = self.buckets[self._index(key)]
        for i in range(len(bucket)):
            if bucket[i][0] == key:
                del bucket[i]
                self.count -= 1
                return True
        return False

    def show(self):
        for i in range(self.size):
            print(f"slot {i}: {self.buckets[i]}")

The pair `(key, value)` is a **tuple**: a small fixed group of values written in round brackets. The underscore at the start of `_index` is a Python convention meaning "this method is for internal use by the class".

In [ ]:
ages = HashTable()
ages.put("cat", 4)
ages.put("dog", 7)
ages.put("horse", 12)
ages.put("listen", 1)
ages.put("silent", 2)        # collides with "listen": both are kept
ages.put("dog", 8)           # an existing key: the value is updated

ages.show()
print()
print("dog    ->", ages.get("dog"))
print("silent ->", ages.get("silent"))
print("zebra  ->", ages.get("zebra"))
print("Items stored:", ages.count)

> ⚠️ **Common pitfalls**
>
> - Forgetting to check whether the key already exists in `put`. The table would then hold two entries for the same key.
> - Writing `[[]] * size` to create the buckets. That produces `size` references to **the same** list, so every key would land in one shared bucket. The comprehension `[[] for i in range(size)]` creates separate lists.

## 5. Why it is fast, and when it is not

A lookup computes an index and then searches one bucket. The computation is O(1), so everything depends on **the buckets being short**. Two things can make them long: a poor hash function, and too few slots.

### Condition 1: the hash function must spread the keys
Let us store 10,000 keys in a table with 20,000 slots. There is plenty of room, so in an ideal world almost every key would have a slot to itself.

In [ ]:
table = HashTable(20_000)
for i in range(10_000):
    table.put(f"key{i}", i)

used = sum(1 for bucket in table.buckets if len(bucket) > 0)
longest = max(len(bucket) for bucket in table.buckets)
print(f"Slots in use: {used} of {table.size}")
print(f"Longest bucket: {longest} items")

A disappointing result. The keys are crowded into a tiny fraction of the slots, in buckets hundreds of items long, while almost all of the table stays empty. The fault lies with `simple_hash`: sums of character codes all fall within a narrow range of numbers.

Python has a carefully designed hash function built in, called `hash()`. To use it, our table needs only a different `_index` method.

In [ ]:
class BetterHashTable(HashTable):
    def _index(self, key):
        return hash(key) % self.size        # Python's own hash function

table = BetterHashTable(20_000)
for i in range(10_000):
    table.put(f"key{i}", i)

used = sum(1 for bucket in table.buckets if len(bucket) > 0)
longest = max(len(bucket) for bucket in table.buckets)
print(f"Slots in use: {used} of {table.size}")
print(f"Longest bucket: {longest} items")

Now the keys are spread across thousands of slots and no bucket holds more than a handful of items.

> 🧠 `BetterHashTable` is three lines long because of **inheritance** (Notebook 07): it reuses `put`, `get` and `remove` unchanged and overrides only the method that computes the index.

### Condition 2: there must be enough slots
The **load factor** measures how full the table is:

$$\text{load factor} = \frac{\text{number of items}}{\text{number of slots}}$$

It is the **average bucket length**. With a load factor below 1, a lookup examines one or two items: effectively **O(1)**. If we squeeze 10,000 items into 10 slots, each bucket holds about 1,000 items and every lookup becomes a long linear search: back to **O(n)**.

### Experiment: the effect of the table size
We store the same 10,000 keys in tables of four different sizes, and time 2,000 random lookups in each.

In [ ]:
import time
import random

n = 10_000
lookups = [f"key{random.randint(0, n - 1)}" for i in range(2_000)]

for size in [10, 100, 1_000, 20_000]:
    table = BetterHashTable(size)
    for i in range(n):
        table.put(f"key{i}", i)

    start = time.perf_counter()
    for key in lookups:
        table.get(key)
    elapsed = time.perf_counter() - start

    print(f"{size:>6} slots: load factor {n / size:>7.1f}, 2,000 lookups in {elapsed:.4f} seconds")

### Observed result
The same data, the same code, the same hash function, and a difference of dozens of times, decided only by the number of slots.

### Resizing
A real hash table therefore **grows**. When the load factor passes a threshold (often around 0.7), it creates a larger list of buckets and re-inserts every item, since each index depends on the size. That one operation is O(n), but it happens so rarely that the average cost of an insertion remains O(1). One of the exercises asks you to implement it.

## 6. Python's `dict` and `set` are hash tables

Everything you have just built is what happens inside Python every time you write `d[key]`. A `dict` is a highly optimised hash table that resizes itself automatically. A `set` is the same structure storing **keys only**, with no values.

This explains three facts we previously had to accept on trust.

- **Lookup, insertion and deletion are O(1) on average**, whatever the size.
- **Dictionary keys must be immutable.** The index is computed from the key. If the key could change after insertion, its index would change and the item could never be found again. That is why a list cannot be a key, while a string, a number or a tuple can.
- **`x in my_set` is fast and `x in my_list` is slow.** The set computes where `x` must be. The list has to look everywhere.

In [ ]:
phone_book = {"Anna": "555-1234", "Kostas": "555-5678"}
print(hash("Anna") % 8)          # the kind of calculation Python does internally

point = (3, 5)
locations = {point: "treasure"}  # a tuple is immutable: allowed as a key
print(locations[(3, 5)])

# locations[[3, 5]] = "x"   ->  TypeError: unhashable type: 'list'

The value printed by `hash("Anna") % 8` changes each time the notebook is restarted. Python deliberately mixes a random value into the hashing of text, as a security measure. Within one session the function is perfectly deterministic.

## 7. Using hash tables to make algorithms faster

Knowing that membership tests and lookups are O(1) is one of the most practical pieces of knowledge in programming. Again and again, an O(n²) solution with a nested loop becomes O(n) once a dictionary or a set **remembers what has been seen**.

### A worked example: two numbers that add up to a target
Given a list of numbers, find two of them whose sum equals a target.

- **The obvious solution** tries every pair: a loop inside a loop, O(n²).
- **The hash-table solution** makes one pass. For each number it asks: *"have I already seen the number that would complete the sum?"* That question is a set lookup, O(1).

In [ ]:
def two_sum_slow(numbers, target):
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            if numbers[i] + numbers[j] == target:
                return numbers[i], numbers[j]
    return None

def two_sum_fast(numbers, target):
    seen = set()
    for num in numbers:
        needed = target - num
        if needed in seen:              # O(1)
            return needed, num
        seen.add(num)
    return None

print(two_sum_slow([4, 9, 1, 7, 12], 16))
print(two_sum_fast([4, 9, 1, 7, 12], 16))

In [ ]:
numbers = list(range(3_000))
target = -1                      # no pair exists: the worst case for both

start = time.perf_counter()
two_sum_slow(numbers, target)
slow_time = time.perf_counter() - start

start = time.perf_counter()
two_sum_fast(numbers, target)
fast_time = time.perf_counter() - start

print(f"Nested loops: {slow_time:.4f} seconds")
print(f"With a set:   {fast_time:.6f} seconds")

### The price
Nothing is free. A hash table uses **more memory** than a list, and it keeps its items in **no useful order**: you cannot ask it for the smallest key, the next larger key, or the keys in a range. For those questions we need a structure that keeps its data ordered, which is the subject of the final notebook.

---
## ✏️ Exercises

### Exercise 1 (Hash by hand)
Using the fact that `ord("a")` is 97, `ord("b")` is 98 and `ord("c")` is 99, compute by hand the slot of the key `"cab"` in a table of size 10 with `simple_hash`. Then check with code.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
# 99 + 97 + 98 = 294, and 294 % 10 = 4
print(simple_hash("cab"), simple_hash("cab") % 10)
```
</details>

### Exercise 2 (Find the collisions)
Among the words below, which ones collide in a table of size 10 with `simple_hash`? Print the slot of each word, then group the words by slot using a dictionary.

In [ ]:
words = ["apple", "pear", "plum", "lime", "kiwi", "fig", "date", "melon"]
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
words = ["apple", "pear", "plum", "lime", "kiwi", "fig", "date", "melon"]

by_slot = {}
for word in words:
    slot = simple_hash(word) % 10
    if slot not in by_slot:
        by_slot[slot] = []
    by_slot[slot].append(word)

for slot in sorted(by_slot):
    print(slot, by_slot[slot])
```
</details>

### Exercise 3 (A better hash function)
Write a function `better_hash(key)` that takes the **position** of each character into account, so that anagrams no longer collide. One classic recipe: start with `total = 0`, and for each character do `total = total * 31 + ord(char)`. Check that `"listen"`, `"silent"` and `"enlist"` now give different values, and different slots in a table of size 13.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def better_hash(key):
    total = 0
    for char in key:
        total = total * 31 + ord(char)
    return total

for word in ["listen", "silent", "enlist"]:
    print(f"{word:<7} hash = {better_hash(word)} -> slot {better_hash(word) % 13}")
```

*Multiplying before adding each character makes the result depend on the order of the characters. Now try `% 10` instead of `% 13`: the three words collide again, because 31 leaves a remainder of 1 when divided by 10, so the multiplication has no effect on the last digit. The hash function and the table size must suit each other, which is one reason why table sizes are often prime numbers, and why production hash functions are designed by specialists.*
</details>

### Exercise 4 (A `contains` method and a `keys` method)
Extend `HashTable` through inheritance with two methods: `contains(key)`, which returns `True` or `False`, and `keys()`, which returns a list of all the keys in the table.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
class HashTablePlus(HashTable):
    def contains(self, key):
        bucket = self.buckets[self._index(key)]
        for stored_key, value in bucket:
            if stored_key == key:
                return True
        return False

    def keys(self):
        result = []
        for bucket in self.buckets:
            for stored_key, value in bucket:
                result.append(stored_key)
        return result

table = HashTablePlus()
table.put("cat", 4)
table.put("dog", 7)
print(table.contains("cat"), table.contains("cow"))
print(table.keys())
```

*`contains` looks in one bucket: O(1). `keys` has to visit every bucket: O(n). Notice that the keys come back in no meaningful order.*
</details>

### Exercise 5 (Word count with your own hash table)
Count how many times each word appears in the sentence below, using a `HashTable` instead of a Python dictionary. (Hint: `get` returns `None` for a word not yet seen.)

In [ ]:
sentence = "the cat sat on the mat the cat slept"
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
sentence = "the cat sat on the mat the cat slept"

counts = HashTable()
for word in sentence.split():
    current = counts.get(word)
    if current is None:
        counts.put(word, 1)
    else:
        counts.put(word, current + 1)

for word in ["the", "cat", "mat", "dog"]:
    print(word, counts.get(word))
```
</details>

### Exercise 6 (First non-repeating character)
Write a function `first_unique(text)` that returns the first character that appears **exactly once** in a string, or `None`. For `"swiss"` the answer is `"w"`. Solve it in O(n) with a dictionary: one pass to count, one pass to find.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def first_unique(text):
    counts = {}
    for char in text:
        counts[char] = counts.get(char, 0) + 1
    for char in text:
        if counts[char] == 1:
            return char
    return None

print(first_unique("swiss"))
print(first_unique("aabb"))
```
</details>

### Exercise 7 (Common elements)
Write a function `common_elements(list1, list2)` that returns the values that appear in both lists, in O(n) rather than O(n²). Test it on two lists of 20,000 numbers and time it.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
import time

def common_elements(list1, list2):
    lookup = set(list1)                  # O(n) to build
    result = []
    for value in list2:
        if value in lookup:              # O(1) each
            result.append(value)
    return result

a = list(range(0, 40_000, 2))            # even numbers
b = list(range(0, 60_000, 3))            # multiples of 3

start = time.perf_counter()
common = common_elements(a, b)
print(f"{len(common)} common values found in {time.perf_counter() - start:.4f} seconds")
print(common[:5])
```
</details>

### Exercise 8 (A table that resizes itself, a little harder)
Write a class `ResizingHashTable` that inherits from `HashTable`. Override `put` so that, after adding an item, it checks the load factor: if `count / size` is greater than 0.7, it doubles the number of buckets and re-inserts every item. Start with 4 slots, insert 20 items, and print the size each time it changes. (Hint: `super().put(key, value)` calls the original `put`.)

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
class ResizingHashTable(HashTable):
    def _index(self, key):
        return hash(key) % self.size

    def put(self, key, value):
        super().put(key, value)
        if self.count / self.size > 0.7:
            self._resize()

    def _resize(self):
        old_buckets = self.buckets
        self.size = self.size * 2
        self.buckets = [[] for i in range(self.size)]
        self.count = 0
        for bucket in old_buckets:
            for key, value in bucket:
                super().put(key, value)      # re-insert: every index changes
        print(f"Resized to {self.size} slots")

table = ResizingHashTable(4)
for i in range(20):
    table.put(f"key{i}", i)

print("Items:", table.count, "| Slots:", table.size)
print(table.get("key13"))
```

*Every item must be re-inserted, because its index depends on `size`. Doubling keeps resizes rare: 20 insertions triggered only a few of them. This is essentially what Python's `dict` does behind the scenes.*
</details>

## 🔑 Key takeaways

- A **hash table** does not search for a key. It **computes** where the key must be.
- A **hash function** turns a key into a number; `% size` turns that number into an index.
- **Collisions** are unavoidable. **Chaining** resolves them by keeping a small list (a bucket) in each slot.
- Lookups are **O(1) on average**, provided the **load factor** stays low and the hash function spreads keys well. Otherwise they degrade towards O(n).
- Real hash tables **resize** themselves as they fill.
- Python's `dict` and `set` are hash tables. That is why their keys must be **immutable**.
- A dictionary or a set that remembers what has been seen often turns an **O(n²)** algorithm into an **O(n)** one.
- The price: more memory, and **no ordering** of the keys.

---
**Next:** *Notebook 18: Trees & Binary Search Trees*, the structure that keeps data ordered **and** fast to search.

---
*© Ioannis Tsioulis. *From Zero to Data Structures & Algorithms in Python*. Prepared for educational use.*